# Parsing PDF-MHBs into JSON for general universities in Germany
In order to convert PDFs into JSON and extract all the necessary information out of it, different approaches are viable.
1. Extracting data using NER-models by cutting the PDF, that was converted to MD using LLM, into blocks and then classifying which block means what
2. Extracting data using LLMs, either directly or after OCR with LLM e.g. Docling
3. Extracting data using Agent, either directly or after OCR to MD using LLM e.g. Docling or Mineru
4. Using XML, Regex and NER-models combined with LLMs and agents since most MHBs are very similar as the tool for generating them (flexnow) has a monopoly in this area

In this case option 3 will be used

Installing all needed dependencies:

In [ ]:
%%capture
!pip3 install docling
!pip3 install 'smolagents[toolkit]'
!pip3 install tokenizers
!pip3 install pydantic

In [ ]:
import os
import sys
os.environ['PYTHONPATH'] = os.path.expanduser('~/mhbai')
sys.path.append(os.path.expanduser('~/mhbai'))

In [ ]:
base_path = "/home/gatterle/mhbai/"
page_break = "---Page Break---"
token_input_percentage = 0.3

USER-INPUT:
Please choose the path to a pdf

In [ ]:
path_to_pdf = "pdfs/20.pdf"
save_files = True
printing = True

## 1. Converting PDF into .md using docling (mineru is an alternative)

### 1.1 Convert PDF to Markdown

In [ ]:
from docling.document_converter import DocumentConverter

path = base_path + path_to_pdf

# create converter
converter = DocumentConverter()

# convert document to markdown
result = converter.convert(path)
md_doc = result.document.export_to_markdown(page_break_placeholder=page_break)

if save_files:
    with open("test-docling.md", "w") as f:
        f.write(md_doc)
if printing:
    print(md_doc)

In [ ]:
# alternatively load previously saved markdown
with open("test-docling.md", "r") as f:
    md_doc = f.read()

### 1.2 Split document into module-chunks

In [ ]:
pages = md_doc.split("\n" + page_break + "\n")

### 1.2.3 University of Augsburg specific fine tuning

__this will be done using NER-models like Bert__

Generate Table of Contents (ToC)

In [ ]:
toc = [pages[1]]
for index, i in enumerate(pages[2:]):
    if "\n| Modul" in i[:15]:
        break
    toc.append(i)

toc = "\n\n".join(e.strip() for e in toc)
pages = pages[index + 2 :]

Generate modules

In [ ]:
# initialize list of modules
modules = []

# group pages into modules
mod = []
for i in pages:
    if "\n| Modul" in i[:15]:
        modules.append(mod)
        mod = [i]
    else:
        mod.append(i)
modules.append(mod)

# combine module pages to single module string and remove empty modules
modules = ["\n\n".join(e.strip() for e in i) for i in modules]
modules = [i for i in modules if i.strip() != ""]

# remove duplicates while preserving order
mods = []
for i in modules:
    if i not in mods:
        mods.append(i)
modules = mods

## 2. Extracting important information using smolagents
1. Creating dataclasses, tools and a prompt
2. Executing smolagents with the previously created md-data

### 2.1.1 Create tools

In [26]:
import json
from ai.information_extraction.data_template import exam_info, time_info, ModuleInfo, ModuleHandbook
if printing:
    print(json.dumps(ModuleHandbook.model_json_schema(), indent=2))

{
  "$defs": {
    "ModuleInfo": {
      "description": "module info class for setting strict type requirements for ai model when extracting information\nabout a module\n\nAttributes:\n    title (str | None): title of the module; often it is found under names like Modulcode\n    module_code (str | None): unique module code; often it is found under names like Titel\n    ects (int | None): number of ECTS credits; often it is found under names like ECTS, LP, CP, Credit Points, Leistungspunkte\n    lecturer (str | None): name of the lecturer; often it is found under names like Dozent, Professor, Modulverantwortlicher, Lehrperson, Dozent, Dozent*in\n    contents (list[str] | None): list of module contents; often it is found under names like Inhalt, Beschreibung\n    goals (list[str] | None): list of learning goals; often it is found under names like Ziele, Lernziele, Kompetenzen\n    requirements (list[str] | None): list of prerequisites; often it is found under names like Anforderungen, Vo

### 2.1.2 Create prompt

In [ ]:
from ai.information_extraction.smolagent import prompt

full_prompt = prompt.format(module_handbook=md_doc)

### 2.1.3 Create agent

In [ ]:
from smolagents import CodeAgent, LiteLLMModel
from smolagents.default_tools import PythonInterpreterTool, FinalAnswerTool
from ai.information_extraction.tools import ValidateOutput


# model = LiteLLMModel(model_id="ollama_chat/gemma4:31b")
# model_name = "qwen3.6:35b-a3b"
# model_name = "gemma4:31b"
model_name = "qwen3.5:4b"
context_window = 256000
model = LiteLLMModel(model_id=f"ollama_chat/{model_name}")

# Create an agent with no tools
agent = CodeAgent(
    tools=[PythonInterpreterTool(), FinalAnswerTool(), ValidateOutput()],
    additional_authorized_imports=["json", "pydantic", "typing", "ai.information_extraction.data_template", "ai.information_extraction.tools"],
    model=model)


### 2.1.4 Create inputs
In order to use the size of the context window as best as possible, use the tokenizer from the used ai model for approximating the size a specific input would have in the context window

In [ ]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("gemma-4-31b-tokenizer.json")

mods = [(i, len(tokenizer.encode(i).ids)) for i in modules]
inputs = []
inp = []
inp_size = 0

limit = token_input_percentage * context_window
limit = 5000
for i in mods:
    if inp_size + i[1] < limit:
        inp.append(i[0])
        inp_size += i[1]
    else:
        inputs.append("\n\n--- NEUES MODUL ---\n\n".join(e.strip() for e in inp))
        inp = [i[0]]
        inp_size = i[1]
inputs.append("\n\n--- NEUES MODUL ---\n\n".join(e.strip() for e in inp))

### 2.2 Run agent

In [ ]:

model = LiteLLMModel(model_id="ollama_chat/qwen3.5:9b")

In [24]:
results = []
for i in inputs:
    agent = CodeAgent(
        tools=[PythonInterpreterTool(), FinalAnswerTool(), ValidateOutput()],
        additional_authorized_imports=["json", "pydantic", "typing", "ai.information_extraction.data_template", "ai.information_extraction.tools"],
        model=model)

    full_prompt = prompt.format(module_handbook=i)
    result = agent.run(full_prompt)
    results.append(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Du bist ein Extraktionsmodell für Modulhandbücher.                                                              │
│                                                                                                                 │
│ ## Aufgabe                                                                                                      │
│ Extrahiere strukturierte Informationen aus dem gegebenen Modulhandbuch und gib ein gültiges                     │
│ ModuleHandbook-Objekt zurück.                                                                                   │
│                                                                                                                 │
│ ## WICHTIGSTE REGELN                                                                                            │
│ 1. Verwende ausschließlich Informationen aus dem Text.                                                          │
│ 2. Erfinde keine Informationen.                                                                                 │
│ 3. Wenn Informationen fehlen → setze Feld auf null.                                                             │
│ 4. Teilweise Informationen dürfen übernommen werden.                                                            │
│ 5. Wiederholungen sind erlaubt, wenn sie sinnvoll sind.                                                         │
│ 6. Fokus liegt auf Korrektheit, nicht auf Vollständigkeit.                                                      │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ## FORMAT DER AUSGABE                                                                                           │
│ - Gib **nur ein JSON-Objekt** im Format von ModuleHandbook zurück.                                              │
│ - KEIN Python-Code                                                                                              │
│ - KEINE Erklärungen                                                                                             │
│ - KEIN zusätzlicher Text                                                                                        │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ## ROBUSTHEITSREGELN (SEHR WICHTIG)                                                                             │
│ - Wenn du unsicher bist → lieber null als raten                                                                 │
│ - Wenn ein Feld schwer zu extrahieren ist → überspringen                                                        │
│ - Wenn Struktur unklar ist → bestmögliche Annäherung wählen                                                     │
│ - Das Ziel ist ein **valider und möglichst vollständiger Output**, nicht Perfektion                             │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ## STRUKTURHINWEISE                                   

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/home/gatterle/mhbai/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='Looking ...er_specific_fields=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...r_specific_fields=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  modules_data = {                                                                                                 
      "modules": []                                                                                                
  }                                                                                                                
                                                                                                                   
  # Module 1: MRM-0002: Statistik                                                                                  
  modules_data["modules"].append({                                                                                 
      "title": "Statistik",                                                                                        
      "module_code": "MRM-0002",                                                                                   
      "ects": 5,                                                                                                   
      "lecturer": "Prof. Dr. Andreas Rathgeber",                                                                   
      "contents": [                                                                                                
          "Zugehörige Veranstaltung: Stochastik (für WING)",                                                       
          "I. Deskriptive Statistik",                                                                              
          "Einführung",                                                                                            
          "Grundbegriffe der Datenerhebung",                                                                       
          "Auswertungsmethoden für ein- und mehrdimensionales Datenmaterial",                                      
          "II. Wahrscheinlichkeitsrechnung",                                                                       
          "Kombinatorische Grundlagen",                                                                            
          "Zufallsvorgänge, Ereignisse und Wahrscheinlichkeiten",                                                  
          "Zufallsvariablen, Verteilungen und Verteilungsparameter",                                               
          "Gesetz der großen Zahlen und zentraler Grenzwertsatz",                                                  
          "III. Induktive Statistik",                                                                              
          "Grundlagen der induktiven Statistik",                                                                   
          "Punkt-Schätzung",                                                                                       
          "Signifikanztests"                                                                                       
      ],                                                                                                           
      "goals": [                                                                                                   
          "Bei vielen wirtschaftswissenschaftlichen Problemstellungen ist die Auswertung von Daten und die         
  Weiterverwendung der Auswertungsergebnisse unerlässlich",                                                        
          "Lernen der theoretischen Grundlagen sowie der Anwendungsvoraussetzungen der statistischen Verfahren",   
          "Anwendung dieser Verfahren im Mittelpunkt stehen",                                                      
          "Einstieg in das empirische Arbeiten erleichtern",                                                       
          "Durchführung eigener Datenauswertungen befähigen",                                                      
          "Gewonnene Ergebnisse interpretieren",         

Execution logs:
Module 1 (MRM-0002) extracted successfully

Out: None

...

[Step 10: Duration 3.55 seconds| Input tokens: 195,457 | Output tokens: 8,489]

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Du bist ein Extraktionsmodell für Modulhandbücher.                                                              │
│                                                                                                                 │
│ ## Aufgabe                                                                                                      │
│ Extrahiere strukturierte Informationen aus dem gegebenen Modulhandbuch und gib ein gültiges                     │
│ ModuleHandbook-Objekt zurück.                                                                                   │
│                                                                                                                 │
│ ## WICHTIGSTE REGELN                                                                                            │
│ 1. Verwende ausschließlich Informationen aus dem Text.                                                          │
│ 2. Erfinde keine Informationen.                                                                                 │
│ 3. Wenn Informationen fehlen → setze Feld auf null.                                                             │
│ 4. Teilweise Informationen dürfen übernommen werden.                                                            │
│ 5. Wiederholungen sind erlaubt, wenn sie sinnvoll sind.                                                         │
│ 6. Fokus liegt auf Korrektheit, nicht auf Vollständigkeit.                                                      │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ## FORMAT DER AUSGABE                                                                                           │
│ - Gib **nur ein JSON-Objekt** im Format von ModuleHandbook zurück.                                              │
│ - KEIN Python-Code                                                                                              │
│ - KEINE Erklärungen                                                                                             │
│ - KEIN zusätzlicher Text                                                                                        │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ## ROBUSTHEITSREGELN (SEHR WICHTIG)                                                                             │
│ - Wenn du unsicher bist → lieber null als raten                                                                 │
│ - Wenn ein Feld schwer zu extrahieren ist → überspringen                                                        │
│ - Wenn Struktur unklar ist → bestmögliche Annäherung wählen                                                     │
│ - Das Ziel ist ein **valider und möglichst vollständiger Output**, nicht Perfektion                             │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ## STRUKTURHINWEISE                                   

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[Step 1: Duration 24.59 seconds]

KeyboardInterrupt: 

Validate output and convert it to JSON

In [25]:
res = []
for result in results:
    try:
        result = ModuleHandbook.model_validate(result)
        res.append(result)
    except Exception as e:
        print(f"Validation failed: {e}")

data = [r.model_dump() for r in res]
modules = []
for d in data:
    modules.extend(mods if (mods := d["modules"]) is not None else [])
if printing:
    print(f"Extracted {len(modules)} modules.")
    print(f"{len([i for i in data if i['modules'] is not None])} of {len(results)} extractions were successful.")

Validation failed: 1 validation error for ModuleHandbook
  Input should be a valid dictionary or instance of ModuleHandbook [type=model_type, input_value='{\n  "modules": [\n    {...n      ]\n    }\n  ]\n}', input_type=AgentText]
    For further information visit https://errors.pydantic.dev/2.13/v/model_type
Extracted 0 modules.
0 of 1 extractions were successful.


In [ ]:
import json

if printing:
    print(json.dumps(data, indent=4, ensure_ascii=False))

## 3. Create confidence-score using Regex and NER-models

In [ ]:
pass

## 4. Save data to database (unia.modules_ai_extracted)

In [ ]:
technique = f"agent-{model_name}"



In [ ]:
print(len(md_doc.split(" ")))